# E4 · Word2Vec entrenado por el equipo
**Parte de:** E0.

**Hipótesis.** El TF-IDF trata cada palabra como independiente: *zorra* y *guarra* no comparten nada. Word2Vec aprende un vector por palabra a partir de su contexto, así que palabras usadas de forma parecida quedan cerca. Esto podría generalizar a insultos o grafías que aparecen pocas veces.

**Cumplimiento de las reglas.** La regla de TweetToxic permite Word2Vec **si lo entrena el equipo**. El modelo se entrena con `gensim` desde cero, **solo con los textos de la partición de entrenamiento** de cada fold (ni la validación ni el test entran en el ajuste). No se usa ningún vector preentrenado.

**Representación.** Promedio y máximo de los vectores de las palabras del tweet (2 × 100 dimensiones), estandarizados. Configuración: skip-gram, ventana 5, `min_count=2`, 40 épocas, semilla 0, 1 hilo (reproducible). Se prueban dos modelos:
- **LR-w2v**: LR sobre Word2Vec + variables de léxico.
- **LR-w2v+tfidf**: la LR original con el bloque Word2Vec agregado.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **Word2Vec** | Algoritmo que aprende un vector numérico por palabra a partir de las palabras que la rodean |
| **Skip-gram** | Variante de Word2Vec que predice el contexto a partir de la palabra; funciona mejor con pocos datos |
| **Embedding** | Vector denso que representa una palabra o un texto |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


## 1. Vecinos más cercanos (Word2Vec entrenado con todo el train, solo para inspección)

In [2]:
from gensim.models import Word2Vec
toks = [tb.RE_TOKEN.findall(t) for t in d.TXT_TR]
w2v = Word2Vec(sentences=toks, **tb.CFG_BASE['W2V'])
print('Vocabulario:', len(w2v.wv))
pd.DataFrame({w: [f'{v} ({s:.2f})' for v, s in w2v.wv.most_similar(w, topn=6)]
              for w in ['zorra', 'moros', 'callate', 'inmigrantes', 'mujeres'] if w in w2v.wv})

Vocabulario: 1977


,zorra,moros,callate,inmigrantes,mujeres
0,perra (0.78),putos (0.92),cllate (0.80),pateras (0.79),esas (0.82)
1,sucia (0.72),negros (0.82),tens (0.78),ilegales (0.76),denuncias (0.81)
2,sirvienta (0.71),trabajar (0.80),pija (0.78),control (0.75),peliculas (0.80)
3,asquerosa (0.69),nuestro (0.80),hable (0.76),pedir (0.75),esclavas (0.80)
4,sepas (0.69),vienen (0.79),horrible (0.75),rescatados (0.74),sexuales (0.79)
5,pendeja (0.68),p (0.78),alv (0.75),llegada (0.74),falsas (0.78)


## 2. Validación cruzada repetida

In [3]:
R = {}
for b in ['LR', 'LR-w2v', 'LR-w2v+tfidf']:
    R[b] = tb.evaluar(d, {s: tb.oof(d, b, s) for s in tb.SEEDS}); tb.log('OOF listo:', b)
for k, bs in {'E0 · stacking 5 modelos': tb.BASES_ORIGINALES,
              '+ LR-w2v': tb.BASES_ORIGINALES + ['LR-w2v'],
              '+ LR-w2v+tfidf': tb.BASES_ORIGINALES + ['LR-w2v+tfidf']}.items():
    R['Stacking ' + k] = tb.evaluar(d, {s: tb.stacking(d, bs, s, con_test=False)[0] for s in tb.SEEDS})
tb.tabla(R)

[     4s] OOF listo: LR


[     4s] OOF listo: LR-w2v


[     4s] OOF listo: LR-w2v+tfidf


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
LR,0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
LR-w2v,0.7479,0.7851,0.7568,0.7633,0.0058,0.7050,0.756 / 0.763 / 0.770
LR-w2v+tfidf,0.7543,0.7771,0.7611,0.7642,0.0008,0.6996,0.763 / 0.765 / 0.765
Stacking E0 · stacking 5 modelos,0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
Stacking + LR-w2v,0.8036,0.8329,0.7976,0.8114,0.0013,0.7730,0.813 / 0.810 / 0.811
Stacking + LR-w2v+tfidf,0.8044,0.8338,0.7961,0.8114,0.0014,0.7751,0.813 / 0.810 / 0.811


## Conclusión: **no se adopta**
- Word2Vec solo llega a un F1 medio de **0,763**, 4 puntos por debajo de la LR con TF-IDF. Con 1.400 tweets por partición (~25 mil palabras) los vectores son ruidosos: los vecinos de *zorra* o *moros* mezclan palabras temáticas y frecuentes, no sinónimos.
- Agregarlo a la LR o al stacking **no mejora** (0,811 frente a 0,813).
- Una mejora posible es entrenar Word2Vec con un corpus grande de tweets en español sin etiquetas (permitido si se cita). No se hizo por falta de un corpus con licencia clara disponible a tiempo, y queda como trabajo futuro.